# Reconstruct frame table from HAL config + shutter file

The inverse of `prepare_imaging/01_create_hal_config_and_shutters`. Given a HAL
config XML and its shutter XML, rebuild the **frame table** (`color`, `channel`,
`z` — one row per camera frame, blanks included).

Use this to **recover a lost `frame_table_*.csv`** or to **verify** that a HAL
config and its shutter file are mutually consistent.

- Per-frame `z` comes from the HAL config `<z_offsets>`.
- Per-frame `channel` comes from the shutter `<event>` list (frames with no event are blank).
- `color` is recovered from `channel` via the microscope's channel map.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import reconstruct_frame_table, read_shutter_reference
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.visualization       import visualize_shutter_sequence

print(f"MERCI_DIR  : {MERCI_DIR}")
print(f"SAMPLE_DIR : {SAMPLE_DIR}")

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE = "MF3"

# ── Choose the HAL config to reconstruct from ──────────────────────────────
# Set HAL_CONFIG to a hal-config-*.xml in settings/. The matching shutter file
# is found automatically from the config's <shutters> element. Set SHUTTER_PATH
# to override the auto-resolution.
HAL_CONFIG   = SETTINGS_DIR / "hal-config-mf3-bits-blkf4-488f2-560f25-650f25-750f25.xml"
SHUTTER_PATH = None        # None -> auto-resolve from the HAL config

# ── Available HAL configs (for reference) ──────────────────────────────────
print("HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

print(f"\nSelected HAL config : {HAL_CONFIG.name}")
print(f"Shutter (referenced): {read_shutter_reference(HAL_CONFIG)}")

## Reconstruct

In [ ]:
frame_table = reconstruct_frame_table(HAL_CONFIG, SHUTTER_PATH, microscope=MICROSCOPE)

# Recover the sequence name from the shutter filename (keeps any -seq suffix).
name = Path(read_shutter_reference(HAL_CONFIG)).stem
if name.startswith("shutter-"):
    name = name[len("shutter-"):]
print(f"Reconstructed frame table: {name}  ({len(frame_table)} frames)")

print_frame_table(frame_table)

visualize_shutter_sequence(
    frame_table,
    title=f"Reconstructed: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## (Optional) Validate against an existing frame table

In [ ]:
existing_csv = METADATA_DIR / f"frame_table_{name}.csv"
if existing_csv.exists():
    existing = pd.read_csv(existing_csv, index_col=0)
    matches = (len(existing) == len(frame_table)) and all(
        np.isclose(frame_table[col].to_numpy(dtype=float),
                   existing[col].to_numpy(dtype=float),
                   equal_nan=True).all()
        for col in ["color", "channel", "z"]
    )
    if matches:
        print(f"PASS - reconstruction matches existing {existing_csv.name}")
    else:
        print(f"DIFFERENT - reconstruction does NOT match existing {existing_csv.name}")
        print("Existing:")
        print_frame_table(existing)
else:
    print(f"No existing frame table at {existing_csv.name} - nothing to validate against.")

## Save

In [ ]:
ft_path = METADATA_DIR / f"frame_table_{name}.csv"
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")